# 🚀 MDAAR v2.0: Multi-role Dynamic Architecture for Automated Repositories
### Polyglot Reverse Engineering, Multi-Provider Auto-Failover & Persistent Hybrid RAG
---
**Highlights:**
1. **100% Full-File Codebase Coverage:** No file-count caps or arbitrary truncation — lossless `Map-Reduce` inspection with `SecretSanitizer`.
2. **Gemini Auto Free Chain (`auto-free-chain`):** Cascades from `gemini-3.8-flash` ➔ `gemini-3.6-flash` ➔ `gemini-3.5-flash-lite` ➔ `gemini-3.1-flash-lite` ➔ `gemini-3-flash-preview` ➔ `gemma-4-31b-it` ➔ `gemma-4-26b-a4b-it`.
3. **OpenAI-Compatible Gateway & 2nd-Colab Router (`OpenAI-Gateway`):** Connect to OpenRouter, Cloudflare AI Gateway, Portkey, Groq, or [`free_router_colab.ipynb`](https://colab.research.google.com/github/braadiv/MDAAR/blob/main/free_router_colab.ipynb).
4. **Conditional Interactive Colab Panels (`ipywidgets`):** Shows only the fields needed by your selected options, shares unified credentials once, and pre-loads Colab Secrets cleanly without `AUTO` clutter.
5. **On-Demand Local GPU Dependencies:** `transformers`, `accelerate`, and `bitsandbytes` install automatically only when `Local-Offline` is used.

## 🔑 Colab Secrets & Environment Variables Reference
All interactive panels pre-load values from **Colab Secrets (🔑)** / `os.environ` automatically, hide unrelated provider controls, and share unified credentials across Planner & Executor:

| Colab Secret / Env Var | Purpose & Default |
|:---|:---|
| `MDAAR_PROJECT_NAME` | Project name (default `MySystem`) |
| `MDAAR_GIT_SOURCE_URL` | Source Git repository URL |
| `MDAAR_SOURCE_AUTH` | Source repo auth (`HTTPS` or `SSH`) |
| `MDAAR_BASE_WORKSPACE` | Workspace folder (Drive if mounted) |
| `MDAAR_ENABLE_EXTERNAL_DOCS_REPO` | Enable external shared docs repo (`True` / `False`) |
| `MDAAR_GIT_DOCS_URL` / `MDAAR_DOCS_AUTH` | External docs Git URL & auth (`HTTPS` / `SSH`) |
| `MDAAR_PLANNER_PROVIDER` | Planner provider (`Gemini`, `OpenAI-Gateway`, `Local-Offline`) |
| `MDAAR_PLANNER_GEMINI_MODEL` / `MDAAR_EXECUTOR_GEMINI_MODEL` | Gemini model/chain (default `auto-free-chain` starting at `gemini-3.8-flash`) |
| `MDAAR_PLANNER_OPENAI_MODELS` / `MDAAR_EXECUTOR_OPENAI_MODELS` | Gateway model name per role (or `OPENAI_MODEL`) |
| `MDAAR_PLANNER_LOCAL_MODEL` / `MDAAR_EXECUTOR_LOCAL_MODEL` | Local HuggingFace model ID per role |
| `MDAAR_EXECUTOR_MODE` | `same` (Same as Planner) or `custom` (Dedicated Executor) |
| `MDAAR_EXECUTOR_PROVIDER` | Executor provider when `custom` |
| `GEMINI_API_KEY` / `MDAAR_GEMINI_KEYS` | **Shared:** Google AI Studio keys (supports comma-separated rotation) |
| `MDAAR_OPENAI_BASE_URL` / `MDAAR_ROUTER_URL` / `OPENAI_BASE_URL` | **Shared:** OpenRouter / Cloudflare / Portkey / 2nd-Colab router URL |
| `MDAAR_OPENAI_API_KEY` / `MDAAR_ROUTER_TOKEN` / `OPENROUTER_API_KEY` | **Shared:** Gateway API key / 2nd-Colab Bearer token |
| `MDAAR_QUANTIZATION` / `HF_TOKEN` | **Shared:** Local quantization (`4bit`/`None`) & HuggingFace token |
| `MDAAR_DOCS_PORT` / `MDAAR_RAG_QUESTION` | MkDocs port (`8000`) & default RAG prompt |

## 1️⃣ Environment Setup & Fast MDAAR Install
Mount Google Drive and install lightweight **MDAAR** + `cloudflared`.

In [ ]:
#@title 1. Initialize Runtime, Mount Drive & Install MDAAR { display-mode: "form" }
import os, json, subprocess
from pathlib import Path

DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ Google Drive mounted successfully.')
    except Exception as e:
        print('⚠️ Drive mount notice (using local storage fallback):', e)

print('⏳ Installing MDAAR & bundled skills...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ Installing cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, SkillLoader
skills_count = len(SkillLoader().list_bundled_skills())
print(f'🎉 MDAAR v{__version__} ready with {skills_count} bundled guardrail skills!')

## 2️⃣ Interactive Repository Setup
Run this cell to configure Source & optional External Documentation repositories. Controls hide/show dynamically based on your choices and auto-load from Colab Secrets.

In [ ]:
#@title 2. Repository Configuration Panel & Connection Test { display-mode: "form" }
from mdaar_pipeline import launch_repo_config_ui
repo_ui = launch_repo_config_ui(lang='en', namespace=globals(), test_on_start=True)

## 3️⃣ Interactive Dual-Model & Shared Credentials Panel
Run this cell to select Planner & Executor providers cleanly:
- Unrelated provider fields hide automatically.
- `GEMINI_API_KEY`, `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `QUANTIZATION`, and `HF_TOKEN` live once in a unified Shared section.

In [ ]:
#@title 3. Planner/Executor & Shared Credentials Panel { display-mode: "form" }
from mdaar_pipeline import launch_models_config_ui
models_ui = launch_models_config_ui(pipeline=globals().get('pipeline'), lang='en', namespace=globals())

## 4️⃣ Fail-Fast Code Synchronization & Diff Tracker

In [ ]:
#@title 4. Sync Code and Detect Changes { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 Current Git Commit: ({current_commit[:7] if current_commit else "Local"})')
if changed_files:
    print(f'🔍 Detected {len(changed_files)} changed files since last documented commit:')
    for f in changed_files[:15]:
        print('  •', f)
else:
    print('ℹ️ Full fresh repository inspection will run across all tracked source files.')

## 5️⃣ 100% Full-File Map-Reduce Generation, Quality Gate & `.rag_index/`

In [ ]:
#@title 5. Execute 100% Full-Codebase Scan & Multi-Agent Generation { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

def _show_map_progress(idx, total, module_name, files_desc):
    if idx == 1 or idx == total or idx % 5 == 0:
        print(f'  [{idx}/{total}] Inspecting `{module_name}`: {files_desc[:90]}')

target_changes = None if FORCE_FULL_GENERATION else changed_files
run_stats = pipeline.run_multi_agent_generation(
    changed_files=target_changes,
    force_full=FORCE_FULL_GENERATION,
    progress_callback=_show_map_progress
)

if current_commit:
    pipeline.save_state(current_commit)

print('='*70)
print('📊 Documentation Pipeline Telemetry:')
print(json.dumps(run_stats, indent=2, ensure_ascii=False))
print('='*70)

## 6️⃣ Documentation Portal & Cloudflare Tunnel

In [ ]:
#@title 6. Launch Portal and Push to Remote Docs Repo { display-mode: "form" }
DOCS_PORT = "" #@param {type:"string"}

from mdaar_pipeline import resolve_setting

docs_port = int(resolve_setting(DOCS_PORT, ('MDAAR_DOCS_PORT',), default='8000') or 8000)
tunnel_url = pipeline.serve_and_tunnel(port=docs_port)

if tunnel_url:
    print('='*70)
    print(f'🎉 Interactive Documentation Portal is live on port {docs_port}:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ Server running locally, check tunnel logs.')

sync_docs_res = pipeline.sync_docs_repo()
print('📡 Remote Docs Sync:', sync_docs_res)

## 7️⃣ Hybrid BM25 Q&A Assistant (`.rag_index/`)

In [ ]:
#@title 7. Query Project Documentation & Verified Codebase Facts { display-mode: "form" }
QUESTION = "" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

from mdaar_pipeline import resolve_setting

resolved_q = resolve_setting(QUESTION, ('MDAAR_RAG_QUESTION',), default='What is the system architecture, database schema, and core APIs?')

chat_engine = pipeline.get_chat_engine()
chunks_loaded = chat_engine.load_documents(prefer_saved_index=True)
print(f'📚 Loaded {chunks_loaded} indexed knowledge chunks from `{chat_engine.index_dir}`')
print(f'🔍 Querying for: "{resolved_q}" (Audience: {AUDIENCE})...\n')

answer = chat_engine.ask(resolved_q, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)